# Prompt chaining support reply

This notebook demonstrates prompt chaining by splitting a support workflow into two model calls.

## 1. Setup

To use the OpenAI API, you need to have an API key.

1. Create or log into your [OpenAI account](https://platform.openai.com/)
2. Go to [API Keys](https://platform.openai.com/api-keys)
3. Click `Create new secret key`, name it, copy it immediately
4. Add billing/payment details so the key becomes active

Once you have the API key, export it as an environment variable called `OPENAI_API_KEY`. If this environment variable is not defined, you will be asked for your API key.

In [ ]:
import json
import os
from textwrap import dedent

from openai import OpenAI

client = OpenAI()
DEFAULT_MODEL = os.getenv('MODEL', 'gpt-4o-mini')

In [ ]:
def extract_issue(message, model=DEFAULT_MODEL):
    response = client.chat.completions.create(
        model=model,
        messages=[
            {
                'role': 'system',
                'content': 'Extract the key support fields from a customer message. Return only valid JSON with the keys product, issue, sentiment, urgency, and next_action.',
            },
            {
                'role': 'user',
                'content': 'Customer message:\nThe dashboard keeps logging me out when I switch tabs. I need to sign in again every time.',
            },
            {
                'role': 'assistant',
                'content': json.dumps({
                    'product': 'dashboard',
                    'issue': 'Session expires or resets when switching tabs',
                    'sentiment': 'frustrated',
                    'urgency': 'medium',
                    'next_action': 'Check session persistence and browser lifecycle handling.',
                }, indent=2),
            },
            {
                'role': 'user',
                'content': f'Customer message:\n{message}\n\nReturn only JSON.',
            },
        ],
        temperature=0,
        response_format={'type': 'json_object'},
    )
    content = response.choices[0].message.content or '{}'
    return json.loads(content)


def draft_reply(extracted, model=DEFAULT_MODEL):
    response = client.chat.completions.create(
        model=model,
        messages=[
            {
                'role': 'system',
                'content': 'You are a support agent. Write a concise reply that acknowledges the issue, summarizes the next step, and stays professional and empathetic.',
            },
            {
                'role': 'user',
                'content': 'Use this structured context to draft the reply:\n' + json.dumps(extracted, indent=2) + '\n\nWrite 3 to 4 sentences. Do not mention the JSON fields.',
            },
        ],
        temperature=0.2,
    )
    return response.choices[0].message.content or ''

In [ ]:
message = dedent('''
I keep getting signed out of the app whenever I move between dashboard tabs.
It is happening on both Chrome and Edge, and it is slowing down our team.
''').strip()

extracted = extract_issue(message)
reply = draft_reply(extracted)

print('=== Prompt chaining support reply ===')
print('Customer message:')
print(message)
print('\nStep 1: extracted context')
print(json.dumps(extracted, indent=2))
print('\nStep 2: support reply')
print(reply)